# Élaguer un détecteur d'objets pré-entraîné

SSDlite320 MobileNetV3-Large est un détecteur d'objets fait pour les téléphones : 3,4 millions de paramètres, entraîné par torchvision sur COCO (80 classes d'objets). On va le réduire par élagage, et mesurer ce que l'on y gagne et ce que l'on y perd :

1. Mesurer la qualité : la mAP
2. Mesurer le coût : paramètres, taille, latence
3. Élagage non structuré : jusqu'où ?
4. Élagage structuré avec Torch-Pruning
5. Réentraîner le modèle élagué
6. La perte, classe par classe

Ce notebook est prévu pour un GPU : *Exécution > Modifier le type d'exécution > GPU T4*. Sans GPU, il tourne quand même, en plus petit : 60 images au lieu de 943, et une seule époque de réentraînement, dont on ne peut rien conclure.

## Mise en place

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh coco-val2017-open
%pip install -q "torchmetrics[detection]" "torch-pruning>=1.6,<2"

In [ ]:
import copy
import gzip
import io
import json
import time
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch_pruning as tp
from torch import nn
from torch.nn.utils import prune
from torch.utils.data import DataLoader, Dataset
from torchmetrics.detection import MeanAveragePrecision
from torchvision import tv_tensors
from torchvision.io import ImageReadMode, decode_image
from torchvision.models.detection import (
  SSDLite320_MobileNet_V3_Large_Weights,
  ssdlite320_mobilenet_v3_large,
)
from torchvision.transforms import v2
from torchvision.utils import draw_bounding_boxes

GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
# Sans GPU : 60 images au lieu de 943, et une seule époque de réentraînement
NB_IMAGES = 943 if GPU else 60
NB_EPOQUES = 10 if GPU else 1
TAUX_APPRENTISSAGE = 3e-4
torch.manual_seed(0)
# Un avertissement de Torch-Pruning sans conséquence sur ce modèle
warnings.filterwarnings("ignore", "Maximum recursive depth", module="torch_pruning")
print(
  "GPU"
  if GPU
  else "Pas de GPU : 60 images et une époque, de quoi vérifier le code seulement"
)

Les données (le dossier `coco-val2017-open` de shuuchuu/datasets, 156 Mo) : 943 images de COCO val2017 (Lin et al., 2014), les 1 000 premières par identifiant dont la photo est sous licence CC BY 2.0 ou CC BY-SA 2.0 et qui contiennent au moins un objet, moins les 57 dont l'auteur a restreint la licence depuis, avec leurs annotations (sous licence CC BY 4.0, comme toutes celles de COCO). L'auteur, la licence et la page Flickr de chaque photo sont dans `coco-val2017-open/credits.csv`. Le modèle n'a jamais vu ces images, torchvision l'a entraîné sur COCO train2017 : la première moitié sert à le réentraîner, la seconde à le tester.

In [ ]:
DOSSIER = Path("coco-val2017-open")
coco = json.loads((DOSSIER / "annotations.json").read_text())
annotations = defaultdict(list)
for annotation in coco["annotations"]:
  annotations[annotation["image_id"]].append(annotation)
infos = sorted(coco["images"], key=lambda info: info["id"])[:NB_IMAGES]
dossier_images = DOSSIER / "images"
print(
  len(infos), "images,", sum(len(annotations[i["id"]]) for i in infos), "objets annotés"
)

Le modèle redimensionne toute image en 320 × 320 avant de la traiter : on le fait ici une fois pour toutes, boîtes comprises, ce qui garde les images en mémoire (300 Mo) et accélère l'entraînement. Étirer une image ne change pas l'IoU de deux boîtes, donc ne change pas la mAP.

Chaque élément est une image et sa cible : les boîtes `boxes` en (x1, y1, x2, y2), les classes `labels` (les identifiants de COCO, de 1 à 90) et `iscrowd` (1 pour une boîte qui entoure une foule, que la mAP de COCO ne compte ni comme trouvée ni comme manquée). L'entraînement n'utilise pas ces boîtes de foule, et retourne une image sur deux horizontalement.

In [ ]:
TAILLE = 320


def charge(info: dict, foules: bool) -> tuple[tv_tensors.Image, dict]:
  image = decode_image(str(dossier_images / info["file_name"]), mode=ImageReadMode.RGB)
  sx, sy = TAILLE / info["width"], TAILLE / info["height"]
  gardees = [
    a
    for a in annotations[info["id"]]
    if min(a["bbox"][2:]) >= 1 and (foules or not a["iscrowd"])
  ]
  # Les boîtes de COCO sont en (x, y, largeur, hauteur), en pixels
  boites = [
    [x * sx, y * sy, (x + w) * sx, (y + h) * sy]
    for x, y, w, h in (a["bbox"] for a in gardees)
  ]
  cible = {
    "boxes": tv_tensors.BoundingBoxes(
      torch.tensor(boites).reshape(-1, 4), format="XYXY", canvas_size=(TAILLE, TAILLE)
    ),
    "labels": torch.tensor([a["category_id"] for a in gardees], dtype=torch.int64),
    "iscrowd": torch.tensor([a["iscrowd"] for a in gardees], dtype=torch.int64),
  }
  return tv_tensors.Image(v2.functional.resize(image, [TAILLE, TAILLE])), cible


class Detection(Dataset):
  def __init__(self, elements: list, entrainement: bool = False) -> None:
    self.elements = elements
    retournement = [v2.RandomHorizontalFlip()] if entrainement else []
    self.transformation = v2.Compose(
      [*retournement, v2.ToDtype(torch.float32, scale=True)]
    )

  def __len__(self) -> int:
    return len(self.elements)

  def __getitem__(self, i: int) -> tuple[torch.Tensor, dict]:
    return self.transformation(*self.elements[i])


def assemble(lot: list) -> tuple:
  return tuple(zip(*lot))


moitie = len(infos) // 2
jeu_entrainement = Detection(
  [charge(info, foules=False) for info in infos[:moitie]], entrainement=True
)
jeu_test = Detection([charge(info, foules=True) for info in infos[moitie:]])
chargeur_entrainement = DataLoader(
  jeu_entrainement, batch_size=16, shuffle=True, collate_fn=assemble
)
chargeur_test = DataLoader(jeu_test, batch_size=16, collate_fn=assemble)
print(len(jeu_entrainement), "images d'entraînement,", len(jeu_test), "de test")

Le modèle prend une liste d'images (des tenseurs 3 × H × W de flottants entre 0 et 1) et, en mode évaluation, renvoie pour chacune un dictionnaire : `boxes`, `scores` et `labels`, jusqu'à 300 détections triées par score. Voici celles de score supérieur à 0,5 sur une image de test :

In [ ]:
POIDS_SSD = SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
categories = POIDS_SSD.meta["categories"]  # le nom de chaque identifiant de classe
modele = ssdlite320_mobilenet_v3_large(weights=POIDS_SSD).to(DEVICE).eval()

image, cible = jeu_test[0]
with torch.inference_mode():
  (prediction,) = modele([image.to(DEVICE)])
garde = prediction["scores"] > 0.5
dessin = draw_bounding_boxes(
  v2.functional.to_dtype(image, torch.uint8, scale=True),
  prediction["boxes"][garde].cpu(),
  labels=[categories[i] for i in prediction["labels"][garde].tolist()],
  colors="red",
  width=2,
)
plt.imshow(dessin.permute(1, 2, 0))
plt.axis("off")
plt.show()

## 1. Mesurer la qualité : la mAP

La mAP (*mean average precision*) de COCO : pour chaque classe, l'aire sous la courbe précision-rappel de ses détections, une détection étant juste si son IoU avec une boîte de la même classe dépasse un seuil ; puis la moyenne sur les classes, et sur les seuils d'IoU de 0,5 à 0,95. `map_50` est celle du seul seuil 0,5.

Écrivez `evalue(modele, chargeur, par_classe=False)`, qui passe le modèle en mode évaluation, prédit les images de `chargeur` sous `torch.inference_mode()`, et renvoie le résultat d'une `MeanAveragePrecision` mise à jour lot par lot. Mesurez la mAP du modèle pré-entraîné sur `chargeur_test`. torchvision annonce 21,3 % (0,213) sur tout COCO val2017 : obtenez-vous la même chose sur ces 472 images ?

**Documentation utile**

- [`MeanAveragePrecision`](https://lightning.ai/docs/torchmetrics/stable/detection/mean_average_precision.html)
- [`ssdlite320_mobilenet_v3_large`](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.ssdlite320_mobilenet_v3_large.html)
- [`torch.inference_mode`](https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html)

In [ ]:
# Votre code ici
def evalue(
  modele: nn.Module, chargeur: DataLoader, par_classe: bool = False
) -> dict: ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `modele.eval()`, puis tout le reste sous `with torch.inference_mode():`
- `MeanAveragePrecision(iou_type="bbox", class_metrics=par_classe)` : les boîtes en (x1, y1, x2, y2), son format par défaut ; `par_classe=True` ajoute la précision moyenne de chaque classe (`map_per_class`, pour l'exercice 6)
- `metrique.warn_on_many_detections = False` : le modèle renvoie jusqu'à 300 détections par image, la mAP de COCO ne compte que les 100 meilleures, et `MeanAveragePrecision` le rappelle à chaque lot
- chaque lot de `chargeur` est un tuple d'images et un tuple de cibles : `for images, cibles in chargeur:`
- `modele([image.to(DEVICE) for image in images])` renvoie la liste des prédictions
- `metrique.update(predictions, cibles)` : deux listes de dictionnaires, sur le même appareil ; ramenez les prédictions sur le CPU, où sont les cibles (`{k: v.cpu() for k, v in p.items()}`)
- `metrique.compute()` renvoie un dictionnaire de tenseurs : `float(resultats["map"])`, `float(resultats["map_50"])`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def evalue(modele: nn.Module, chargeur: DataLoader, par_classe: bool = False) -> dict:
  modele.eval()
  metrique = MeanAveragePrecision(iou_type="bbox", class_metrics=...)
  metrique.warn_on_many_detections = False
  with torch.inference_mode():
    for images, cibles in chargeur:
      predictions = modele(...)
      metrique.update([{k: v.cpu() for k, v in p.items()} for p in predictions], ...)
  return metrique.compute()


metriques_base = evalue(modele, chargeur_test)
print(
  f"mAP {float(metriques_base['map']):.3f}, mAP@0.5 {float(metriques_base['map_50']):.3f}"
)
```

### Solution

In [ ]:
def evalue(modele: nn.Module, chargeur: DataLoader, par_classe: bool = False) -> dict:
  modele.eval()
  metrique = MeanAveragePrecision(iou_type="bbox", class_metrics=par_classe)
  # La mAP de COCO ne compte que les 100 meilleures détections par image
  metrique.warn_on_many_detections = False
  with torch.inference_mode():
    for images, cibles in chargeur:
      predictions = modele([image.to(DEVICE) for image in images])
      metrique.update([{k: v.cpu() for k, v in p.items()} for p in predictions], cibles)
  return metrique.compute()


metriques_base = evalue(modele, chargeur_test)
print(
  f"mAP {float(metriques_base['map']):.3f}, mAP@0.5 {float(metriques_base['map_50']):.3f}"
)

## 2. Mesurer le coût : paramètres, taille, latence

Écrivez `profil(modele)`, qui renvoie un dictionnaire :

- `parametres_M` : le nombre de paramètres du modèle, en millions ;
- `taille_Mo` : la taille du fichier de ses poids (`torch.save(modele.state_dict(), ...)`), en Mo ;
- `gzip_Mo` : la taille de ce fichier compressé par gzip, en Mo ;
- `latence_ms` : la latence médiane d'une prédiction d'une seule image (*batch* de 1), en millisecondes, **sur le CPU** : SSDlite est fait pour les téléphones et les processeurs embarqués, son matériel cible. Prédisez `image` 60 fois avec une copie du modèle sur le CPU, et ignorez les 10 premières mesures (la chauffe) ;
- `p95_ms` : le 95ᵉ centile de ces mêmes mesures, en millisecondes.

Une latence se mesure sur une machine au repos : sur Colab, elle varie de quelques pour cent d'une exécution à l'autre.

**Documentation utile**

- [`torch.save`](https://docs.pytorch.org/docs/stable/generated/torch.save.html)
- [`io.BytesIO`](https://docs.python.org/3/library/io.html#io.BytesIO)
- [`gzip.compress`](https://docs.python.org/3/library/gzip.html#gzip.compress)
- [`copy.deepcopy`](https://docs.python.org/3/library/copy.html#copy.deepcopy)
- [`time.perf_counter`](https://docs.python.org/3/library/time.html#time.perf_counter)

In [ ]:
image = jeu_test[0][0]  # l'image des mesures de latence


# Votre code ici
def profil(modele: nn.Module) -> dict: ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `sum(p.numel() for p in modele.parameters())` : le nombre de paramètres, 3,44 millions pour le modèle pré-entraîné
- `tampon = io.BytesIO()`, puis `torch.save(modele.state_dict(), tampon)` : le fichier en mémoire ; `tampon.getvalue()` en donne les octets
- `len(gzip.compress(octets))` : la taille compressée
- `modele_cpu = copy.deepcopy(modele).cpu().eval()` : une copie, pour ne pas déplacer le modèle lui-même
- `debut = time.perf_counter()`, puis `modele_cpu([image])`, puis `time.perf_counter() - debut`, en secondes, sous `torch.inference_mode()`
- `np.median(durees[10:])` et `np.percentile(durees[10:], 95)` : la médiane et le 95ᵉ centile, sans la chauffe

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def profil(modele: nn.Module) -> dict:
  tampon = io.BytesIO()
  torch.save(modele.state_dict(), tampon)
  modele_cpu = copy.deepcopy(modele).cpu().eval()
  durees = []
  with torch.inference_mode():
    for _ in range(60):
      debut = time.perf_counter()
      ...
      durees.append(time.perf_counter() - debut)
  return {
    "parametres_M": sum(p.numel() for p in modele.parameters()) / 1e6,
    "taille_Mo": len(tampon.getvalue()) / 1e6,
    "gzip_Mo": ...,
    "latence_ms": 1000 * np.median(...),
    "p95_ms": 1000 * np.percentile(...),
  }


profil(modele)
```

### Solution

In [ ]:
def profil(modele: nn.Module) -> dict:
  tampon = io.BytesIO()
  torch.save(modele.state_dict(), tampon)
  modele_cpu = copy.deepcopy(modele).cpu().eval()
  durees = []
  with torch.inference_mode():
    for _ in range(60):
      debut = time.perf_counter()
      modele_cpu([image])
      durees.append(time.perf_counter() - debut)
  return {
    "parametres_M": sum(p.numel() for p in modele.parameters()) / 1e6,
    "taille_Mo": len(tampon.getvalue()) / 1e6,
    "gzip_Mo": len(gzip.compress(tampon.getvalue())) / 1e6,
    "latence_ms": 1000 * np.median(durees[10:]),
    "p95_ms": 1000 * np.percentile(durees[10:], 95),
  }


profil(modele)

Le tableau des résultats, que chaque exercice complète d'une ligne :

In [ ]:
resultats = {}


def enregistre(nom: str, modele: nn.Module, metriques: dict) -> pd.DataFrame:
  """Ajoute la ligne du modèle au tableau, et affiche le tableau."""
  resultats[nom] = {
    "mAP": float(metriques["map"]),
    "mAP@0.5": float(metriques["map_50"]),
    **profil(modele),
  }
  return pd.DataFrame(resultats).T.round(3)


enregistre("pré-entraîné", modele, metriques_base)

## 3. Élagage non structuré : jusqu'où ?

Élaguez trois copies du modèle, à 30, 50 et 70 % de parcimonie (la part des poids des couches `nn.Conv2d` et `nn.Linear` mise à zéro), par l'élagage non structuré global des slides, sans réentraînement, et ajoutez la ligne de chacune au tableau. Tracez ensuite la mAP et la taille du fichier compressé (`gzip_Mo`) en fonction de la parcimonie, de 0 (le modèle pré-entraîné) à 70 %. Jusqu'où peut-on élaguer en gardant 90 % de la mAP ? Qu'est-ce qui ne change pas : le nombre de paramètres, la taille du fichier, la latence ? Enfin, calculez la part de poids nuls de chaque couche du modèle élagué à 50 % : le seuil étant commun à tout le réseau, quelles couches perdent le plus de poids ?

**Documentation utile**

- [`torch.nn.Module.named_modules`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.named_modules)
- [`pandas.Series.plot`](https://pandas.pydata.org/docs/reference/api/pandas.Series.plot.html)
- [`matplotlib.axes.Axes.axhline`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.axhline.html)
- [`pandas.Series.sort_values`](https://pandas.pydata.org/docs/reference/api/pandas.Series.sort_values.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- une boucle sur `PARCIMONIES = [0.3, 0.5, 0.7]` ; dans la boucle, le code des slides sur une copie, `non_structure = copy.deepcopy(modele)`, avec `amount=parcimonie` : le modèle pré-entraîné reste intact
- `{n: m for n, m in non_structure.named_modules() if isinstance(m, types_couches)}` : les couches à élaguer, par nom
- `float((m.weight == 0).float().mean())` : la part de zéros d'une couche ; un `pd.Series` du dictionnaire `{nom: part}` pour chaque parcimonie
- `enregistre(f"non structuré {parcimonie:.0%}", non_structure, evalue(non_structure, chargeur_test))` renvoie le tableau, ses lignes dans l'ordre où elles ont été ajoutées, le modèle pré-entraîné en premier
- `tableau[["mAP", "gzip_Mo"]].set_axis([0, *PARCIMONIES])` : les deux colonnes, indexées par la parcimonie ; `.plot(ax=..., marker="o")` sur chacun des deux axes de `plt.subplots(1, 2)`, et `ax.axhline(...)` pour le seuil, 90 % de la mAP du modèle pré-entraîné (`balayage["mAP"].iloc[0]`)
- `sort_values(ascending=False)` : les couches les plus élaguées d'abord

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
PARCIMONIES = [0.3, 0.5, 0.7]
types_couches = nn.Conv2d | nn.Linear
parts_zeros = {}
for parcimonie in PARCIMONIES:
  non_structure = copy.deepcopy(modele)
  couches = {
    n: m for n, m in non_structure.named_modules() if isinstance(m, types_couches)
  }
  poids = [(m, "weight") for m in couches.values()]
  prune.global_unstructured(poids, pruning_method=..., amount=...)
  for module, nom in poids:
    ...
  parts_zeros[parcimonie] = pd.Series(
    {n: float((m.weight == 0).float().mean()) for n, m in couches.items()}
  )
  tableau = enregistre(...)

balayage = tableau[["mAP", "gzip_Mo"]].set_axis([0, *PARCIMONIES])
fig, (gauche, droite) = plt.subplots(1, 2, figsize=(10, 3.5))
balayage["mAP"].plot(ax=gauche, marker="o", xlabel="parcimonie", ylabel="mAP")
gauche.axhline(..., color="gray", linestyle="--", label="90 % de la mAP")
gauche.legend()
balayage["gzip_Mo"].plot(
  ax=droite, marker="o", xlabel="parcimonie", ylabel="taille compressée (Mo)"
)
plt.show()
parts_zeros[0.5].sort_values(...).round(2)
```

### Solution

In [ ]:
PARCIMONIES = [0.3, 0.5, 0.7]
types_couches = nn.Conv2d | nn.Linear
parts_zeros = {}
for parcimonie in PARCIMONIES:
  non_structure = copy.deepcopy(modele)
  couches = {
    n: m for n, m in non_structure.named_modules() if isinstance(m, types_couches)
  }
  poids = [(m, "weight") for m in couches.values()]
  prune.global_unstructured(
    poids, pruning_method=prune.L1Unstructured, amount=parcimonie
  )
  for module, nom in poids:
    prune.remove(module, nom)
  parts_zeros[parcimonie] = pd.Series(
    {n: float((m.weight == 0).float().mean()) for n, m in couches.items()}
  )
  tableau = enregistre(
    f"non structuré {parcimonie:.0%}",
    non_structure,
    evalue(non_structure, chargeur_test),
  )

# Le modèle pré-entraîné, parcimonie 0, puis les trois modèles élagués
balayage = tableau[["mAP", "gzip_Mo"]].set_axis([0, *PARCIMONIES])
fig, (gauche, droite) = plt.subplots(1, 2, figsize=(10, 3.5))
balayage["mAP"].plot(ax=gauche, marker="o", xlabel="parcimonie", ylabel="mAP")
gauche.axhline(
  0.9 * balayage["mAP"].iloc[0], color="gray", linestyle="--", label="90 % de la mAP"
)
gauche.legend()
balayage["gzip_Mo"].plot(
  ax=droite, marker="o", xlabel="parcimonie", ylabel="taille compressée (Mo)"
)
plt.show()
parts_zeros[0.5].sort_values(ascending=False).round(2)

## 4. Élagage structuré avec Torch-Pruning

L'élagage structuré retire des canaux entiers : les tenseurs rétrécissent. Sur une autre copie du modèle (`structure`), retirez 20 % des canaux de chaque couche avec un `tp.pruner.BasePruner` et une `tp.importance.MagnitudeImportance(p=2)`, comme dans les slides (qui en retirent la moitié), en protégeant les têtes de détection (`structure.head`) : leurs sorties, pour chaque ancre les scores des 91 identifiants de classe et les 4 coordonnées d'une boîte, sont fixées par la tâche. Torch-Pruning suit les calculs du modèle sur un exemple d'entrée : ici, une liste d'une image de 3 × 320 × 320 sur `DEVICE`. Affichez la ligne du modèle dans le tableau : que valent la mAP, la taille et la latence ?

**Documentation utile**

- [Torch-Pruning](https://github.com/VainF/Torch-Pruning)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `structure = copy.deepcopy(modele)`
- `exemple = [torch.rand(3, 320, 320, device=DEVICE)]` : le modèle prend une liste d'images
- `tp.pruner.BasePruner(structure, exemple, importance=tp.importance.MagnitudeImportance(p=2), pruning_ratio=0.2, ignored_layers=[structure.head])` : les canaux de plus petite norme L2 partent, 20 % de ceux de chaque couche
- `elagueur.step()` les retire, une fois pour toutes ; dans les têtes, Torch-Pruning retire aussi les canaux d'entrée qui correspondaient aux canaux retirés : seules leurs sorties sont protégées
- `enregistre("structuré 20 %", structure, evalue(structure, chargeur_test))`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
structure = copy.deepcopy(modele)
exemple = [torch.rand(3, 320, 320, device=DEVICE)]
elagueur = tp.pruner.BasePruner(
  structure,
  exemple,
  importance=...,
  pruning_ratio=...,
  ignored_layers=[...],
)
elagueur.step()
enregistre(..., structure, evalue(structure, chargeur_test))
```

### Solution

In [ ]:
structure = copy.deepcopy(modele)
exemple = [torch.rand(3, 320, 320, device=DEVICE)]
elagueur = tp.pruner.BasePruner(
  structure,
  exemple,
  importance=tp.importance.MagnitudeImportance(p=2),
  pruning_ratio=0.2,
  ignored_layers=[structure.head],
)
elagueur.step()
enregistre("structuré 20 %", structure, evalue(structure, chargeur_test))

## 5. Réentraîner le modèle élagué

Écrivez `entraine_epoque(modele, chargeur, optimiseur)` : une époque d'entraînement, qui renvoie la perte moyenne. En mode entraînement (`modele.train()`), le détecteur prend les images **et** leurs cibles, et renvoie ses pertes, un dictionnaire : `bbox_regression` (les boîtes) et `classification` (les classes) ; la perte à minimiser est leur somme. Réentraînez `structure` pendant `NB_EPOQUES` époques sur `chargeur_entrainement`, avec `torch.optim.AdamW` et un taux d'apprentissage de `TAUX_APPRENTISSAGE`, en affichant la perte de chaque époque, puis affichez la ligne du modèle dans le tableau. Quelle part de la mAP perdue regagne-t-on ?

**Documentation utile**

- [`torch.optim.AdamW`](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html)

In [ ]:
# Votre code ici
def entraine_epoque(
  modele: nn.Module, chargeur: DataLoader, optimiseur: torch.optim.Optimizer
) -> float: ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `modele.train()` au début de l'époque (`evalue` repasse le modèle en mode évaluation)
- les images et les cibles sur `DEVICE` : `[image.to(DEVICE) for image in images]`, `[{k: v.to(DEVICE) for k, v in t.items()} for t in cibles]`
- `pertes = modele(images, cibles)`, puis `perte = sum(pertes.values())`
- `optimiseur.zero_grad()`, `perte.backward()`, `optimiseur.step()` ; `perte.item()` pour la moyenne
- `torch.optim.AdamW(structure.parameters(), lr=TAUX_APPRENTISSAGE)`, créé une fois, avant la boucle des époques
- `enregistre("structuré 20 %, réentraîné", structure, evalue(structure, chargeur_test))`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def entraine_epoque(
  modele: nn.Module, chargeur: DataLoader, optimiseur: torch.optim.Optimizer
) -> float:
  modele.train()
  total = 0.0
  for images, cibles in chargeur:
    images = [image.to(DEVICE) for image in images]
    cibles = [{k: v.to(DEVICE) for k, v in t.items()} for t in cibles]
    pertes = modele(...)
    perte = ...
    optimiseur.zero_grad()
    perte.backward()
    optimiseur.step()
    total += perte.item()
  return total / len(chargeur)


optimiseur = torch.optim.AdamW(structure.parameters(), lr=TAUX_APPRENTISSAGE)
for epoque in range(NB_EPOQUES):
  print(epoque + 1, entraine_epoque(structure, chargeur_entrainement, optimiseur))
enregistre(..., structure, evalue(structure, chargeur_test))
```

### Solution

In [ ]:
def entraine_epoque(
  modele: nn.Module, chargeur: DataLoader, optimiseur: torch.optim.Optimizer
) -> float:
  modele.train()
  total = 0.0
  for images, cibles in chargeur:
    images = [image.to(DEVICE) for image in images]
    cibles = [{k: v.to(DEVICE) for k, v in t.items()} for t in cibles]
    pertes = modele(images, cibles)
    perte = sum(pertes.values())
    optimiseur.zero_grad()
    perte.backward()
    optimiseur.step()
    total += perte.item()
  return total / len(chargeur)


optimiseur = torch.optim.AdamW(structure.parameters(), lr=TAUX_APPRENTISSAGE)
for epoque in range(NB_EPOQUES):
  perte = entraine_epoque(structure, chargeur_entrainement, optimiseur)
  print(f"époque {epoque + 1} : perte {perte:.3f}")
enregistre("structuré 20 %, réentraîné", structure, evalue(structure, chargeur_test))

## 6. La perte, classe par classe

Une mAP globale peut cacher des pertes concentrées sur quelques classes : un réseau élagué oublie d'abord les exemples rares et difficiles (Hooker et al., 2019, *What Do Compressed Deep Neural Networks Forget?*). Évaluez le modèle pré-entraîné et le modèle réentraîné avec `par_classe=True`, puis construisez un `DataFrame` indexé par le nom de la classe, avec le nombre d'objets de la classe dans les images de test, la précision moyenne (AP) avant et après l'élagage, et leur écart. Gardez les classes d'au moins `MIN_OBJETS` objets, et triez-les par écart. Toutes les classes perdent-elles autant ? Celles qui perdent le plus sont-elles les plus rares ?

**Documentation utile**

- [`collections.Counter`](https://docs.python.org/3/library/collections.html#collections.Counter)
- [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [`DataFrame.sort_values`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html)

In [ ]:
# Moins d'objets : l'AP de la classe dépend trop des quelques images tirées
MIN_OBJETS = 20 if GPU else 1

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `evalue(modele, chargeur_test, par_classe=True)` et de même pour `structure` : `metriques["map_per_class"]` donne l'AP de chaque classe, dans l'ordre des identifiants de `metriques["classes"]` ; `pd.Series(aps, index=identifiants)` les associe
- les objets de chaque classe : un `Counter` des `labels` des cibles de `jeu_test.elements`, sans les boîtes de foule (`iscrowd` à 1)
- `pd.DataFrame({"objets": ..., "AP avant": ..., "AP après": ...})` aligne les trois séries sur leurs identifiants ; `classes.index = [categories[i] for i in classes.index]` les remplace par les noms
- `classes[classes["objets"] >= MIN_OBJETS].sort_values("écart")` : la classe qui perd le plus en premier

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def ap_par_classe(metriques: dict) -> pd.Series:
  return pd.Series(
    metriques["map_per_class"].tolist(), index=metriques["classes"].tolist()
  )


avant = ap_par_classe(evalue(modele, chargeur_test, par_classe=True))
apres = ap_par_classe(...)
objets = Counter(
  etiquette
  for _, cible in jeu_test.elements
  for etiquette, foule in zip(cible["labels"].tolist(), cible["iscrowd"].tolist())
  if not foule
)
classes = pd.DataFrame(
  {"objets": pd.Series(objets), "AP avant": avant, "AP après": apres}
)
classes["écart"] = ...
classes.index = [categories[i] for i in classes.index]
classes[classes["objets"] >= MIN_OBJETS].sort_values(...).round(3)
```

### Solution

In [ ]:
def ap_par_classe(metriques: dict) -> pd.Series:
  return pd.Series(
    metriques["map_per_class"].tolist(), index=metriques["classes"].tolist()
  )


avant = ap_par_classe(evalue(modele, chargeur_test, par_classe=True))
apres = ap_par_classe(evalue(structure, chargeur_test, par_classe=True))
objets = Counter(
  etiquette
  for _, cible in jeu_test.elements
  for etiquette, foule in zip(cible["labels"].tolist(), cible["iscrowd"].tolist())
  if not foule
)
classes = pd.DataFrame(
  {"objets": pd.Series(objets), "AP avant": avant, "AP après": apres}
)
classes["écart"] = classes["AP après"] - classes["AP avant"]
classes.index = [categories[i] for i in classes.index]
classes[classes["objets"] >= MIN_OBJETS].sort_values("écart").round(3)

## Bilan

Tous les modèles côte à côte :

In [ ]:
pd.DataFrame(resultats).T.round(3)

- Lequel de ces modèles déploieriez-vous sur un téléphone ? Que faudrait-il encore mesurer, et où (sur le téléphone lui-même, avec son moteur d'inférence) ?
- Retirer la moitié des canaux, comme dans les slides, ramène la mAP de ce détecteur à presque 0, et quelques époques sur 471 images n'en regagnent presque rien : un réseau déjà compact comme MobileNetV3 a peu de canaux inutiles.
- Pour regagner plus de mAP : réentraîner plus longtemps, sur COCO train2017 (118 000 images) ; élaguer progressivement, par petites étapes séparées de réentraînements (un plan d'élagage) ; retirer moins de canaux ; distiller le modèle d'origine dans le modèle élagué.
- L'élagage non structuré n'a réduit que le fichier compressé : à quoi peut-il servir quand même ?